## 1. Importamos las librerías necesarias

In [5]:
import pandas as pd
from pathlib import Path
import os
import re

## 2. Definimos rutas relativas y cargamos los 3 datasets

In [6]:
# Definimos rutas relativas para que sea reproducible
DATA_PATH = Path("../data")

# Carga de los 3 datasets
df_a = pd.read_csv(DATA_PATH / "listings_plataforma_A.csv")
df_b = pd.read_csv(DATA_PATH / "listings_plataforma_B.csv")
df_c = pd.read_csv(DATA_PATH / "listings_plataforma_C.csv")

## 3. Limpieza y normalización del Dataset A

In [7]:
df_a

,id_aviso,titulo,zona_barrio,tipo,precio,expensas,superficie_total,superficie_cubierta,ambientes,dormitorios,banios,antiguedad,cochera,balcon,amenities,latitud,longitud,fecha_publicacion
0,A-700480,Departamento en alquiler en Belgrano - 4 ambie...,BELGRANO,Departamento,1.267.000,140298.0,75,75.0,4 amb.,3.0,2.0,4 años,Sí,Sí,NaN,-34.549646421652646,-58.462834001141694,04/01/2025
1,A-220016,Departamento en alquiler en Almagro - 2 ambientes,Almagro,Departamento,584.000,77822.0,39,NaN,2 amb.,1.0,NaN,6,No,No,NaN,-34.60873815826056,-58.43909947532191,15/01/2024
2,A-492142,Departamento en alquiler en Flores - 2 ambientes,Flores,Departamento,252.000,NaN,35 m²,35.0,2,NaN,1.0,23,Sí,No,NaN,"-34,62205081312381",-58.45626733855754,07/03/2024
3,A-675147,PH en alquiler en Belgrano - 2 ambientes,Belgrano,PH,546.000,NaN,38 m²,38.0,2 amb.,1.0,1.0,A estrenar,No,No,Pileta,-34.574756771995716,-58.463757544997684,04/12/2024
4,A-307536,Departamento en alquiler en Villa Urquiza - 1 ...,Villa Urquiza,Departamento,337.000,52942.0,33 m²,NaN,1,NaN,1.0,2,No,Sí,NaN,-34.55428879305003,-58.4866156408398,25/02/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3657,A-309061,Departamento en alquiler en Belgrano - 2 ambie...,Belgrano,Departamento,520.000,NaN,27 m²,27.0,2 amb.,1.0,1.0,3,No,Sí,NaN,-34.55689398126551,-58.45407216308726,26/11/2024
3658,A-279762,Departamento en alquiler en Colegiales - 6 amb...,Colegiales,Departamento,2.516.000,250450.0,131 m²,131.0,6,5.0,4.0,5 años,No,No,NaN,-34.54744066498386,-58.4475401302323,06/12/2024
3659,A-304109,Departamento en alquiler en Flores - 3 ambientes,Flores,Departamento,604.000,98331.0,52 m²,NaN,3,2.0,1.0,30,No,Sí,NaN,-34.64945896983061,-58.46751202334893,10/06/2025
3660,A-741728,Casa en alquiler en Palermo - 6 ambientes,Palermo,Casa,3.102.000,NaN,164,164.0,6 amb.,5.0,3.0,29,No,Sí,Pileta,-34.59781687424598,-58.42537874055159,17/12/2024


## 4. Verificamos duplicados y eliminamos

In [8]:
# 1. Verificamos cuántos duplicados hay antes de borrar
duplicados_antes = df_a['id_aviso'].duplicated().sum()
print(f"Registros duplicados detectados en id_aviso: {duplicados_antes}")

# 2. Eliminamos los duplicados manteniendo la primera aparición
df_a = df_a.drop_duplicates(subset='id_aviso', keep='first')
print(f"Registros en Plataforma A después de limpiar IDs: {len(df_a)}")

Registros duplicados detectados en id_aviso: 3
Registros en Plataforma A después de limpiar IDs: 3659


## 5. Validación de Fechas de Publicación

In [9]:
# 1. Convertimos una sola vez usando .loc para evitar el warning
df_a.loc[:, 'fecha_publicacion'] = pd.to_datetime(df_a['fecha_publicacion'], dayfirst=True, errors='coerce')

# 2. Verificamos errores
errores_fecha = df_a['fecha_publicacion'].isna().sum()
print(f"Fechas con formato incorrecto: {errores_fecha}")

# 3. Analizamos el rango
print(f"Aviso más antiguo: {df_a['fecha_publicacion'].min()}")
print(f"Aviso más reciente: {df_a['fecha_publicacion'].max()}")

Fechas con formato incorrecto: 0
Aviso más antiguo: 2024-01-01 00:00:00
Aviso más reciente: 2025-08-23 00:00:00


## 6. Limpieza integral del Dataset A

In [ ]:
# 1. Cortamos el vínculo inmediatamente para evitar warnings
df_a = df_a.copy() 

# 2. Funciones auxiliares
def normalizar_precio_base(valor):
    if pd.isna(valor): return None
    valor = str(valor).upper().strip()
    es_dolar = any(simbolo in valor for simbolo in ['USD', 'U$D', 'US$'])
    solo_numeros = "".join(filter(str.isdigit, valor))
    if not solo_numeros: return None
    precio = float(solo_numeros)
    return precio * 1415 if es_dolar else precio

# 3. Eliminación de columnas
cols_borrar = ['id_aviso', 'titulo', 'superficie_cubierta', 'dormitorios', 'banios']
df_a = df_a.drop(columns=[c for c in cols_borrar if c in df_a.columns])

# 4. Normalización de barrios
df_a['zona_barrio'] = df_a['zona_barrio'].astype(str).str.strip().str.title()
mapeo_barrios = {
    'V. Crespo': 'Villa Crespo', 'V Crespo': 'Villa Crespo',
    'V. Urquiza': 'Villa Urquiza', 'V Urquiza': 'Villa Urquiza',
    'V. Devoto': 'Villa Devoto', 'V. Del Parque': 'Villa Del Parque',
    'V. Luro': 'Villa Luro', 'Nunez': 'Nuñez', 'Constitucion': 'Constitución'
}
df_a['zona_barrio'] = df_a['zona_barrio'].replace(mapeo_barrios)

# 5. Limpieza de ambientes, superficie y piscina
df_a['ambientes'] = pd.to_numeric(df_a['ambientes'].astype(str).str.replace(r'[^0-9]', '', regex=True), errors='coerce')
df_a['superficie_total'] = pd.to_numeric(df_a['superficie_total'].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')

if 'amenities' in df_a.columns:
    df_a = df_a.rename(columns={'amenities': 'piscina'})

pd.set_option('future.no_silent_downcasting', True)

for col in ['cochera', 'balcon', 'piscina']:
    if col in df_a.columns:
        df_a[col] = df_a[col].fillna(0).apply(lambda x: 1 if x != 0 and str(x).lower() not in ['no', 'nan', 'none', '0', 'false'] else 0)

# 6. Procesamiento de Precio
df_a['precio'] = df_a['precio'].apply(normalizar_precio_base)

# 7. Imputación de Precios
columnas_referencia = ['zona_barrio', 'ambientes']
mediana_barrio_amb = df_a.groupby(columnas_referencia)['precio'].transform('median')
df_a['precio'] = df_a['precio'].fillna(mediana_barrio_amb).fillna(df_a['precio'].median())

# Borramos expensas
df_a = df_a.drop(columns=['expensas'], errors='ignore')

# 8. Antigüedad
df_a['antiguedad'] = df_a['antiguedad'].astype(str).str.lower().str.strip().replace('a estrenar', '0')
df_a['antiguedad'] = pd.to_numeric(df_a['antiguedad'].str.extract(r'(\d+)', expand=False), errors='coerce')
df_a['antiguedad'] = df_a['antiguedad'].fillna(df_a['antiguedad'].median()).astype('Int64')

# 9. Filtro final y limpieza de nombres
# Ajustamos el filtro de precio máximo a algo más coherente
df_a = df_a[(df_a['precio'] >= 100000) & (df_a['precio'] <= 5000000)]
df_a = df_a.rename(columns={'zona_barrio': 'barrio'})
df_a['tipo'] = df_a['tipo'].replace('PH', 'Ph')

df_a.head()

✅ DF_A procesado con éxito: Expensas eliminadas, precio puro de alquiler.


,barrio,tipo,precio,superficie_total,ambientes,antiguedad,cochera,balcon,piscina,latitud,longitud,fecha_publicacion
0,Belgrano,Departamento,1267000.0,75,4,4,1,1,0,-34.549646421652646,-58.462834001141694,2025-01-04 00:00:00
1,Almagro,Departamento,584000.0,39,2,6,0,0,0,-34.60873815826056,-58.43909947532191,2024-01-15 00:00:00
2,Flores,Departamento,252000.0,35,2,23,1,0,0,"-34,62205081312381",-58.45626733855754,2024-03-07 00:00:00
3,Belgrano,Ph,546000.0,38,2,0,0,0,1,-34.574756771995716,-58.463757544997684,2024-12-04 00:00:00
4,Villa Urquiza,Departamento,337000.0,33,1,2,0,1,0,-34.55428879305003,-58.4866156408398,2024-02-25 00:00:00


## 7. Limpieza y normalización del Dataset B

In [11]:
df_b.head()

,listing_id,category_l1,category_l2,property_type,neighborhood,price_ars,covered_area_m2,total_area_m2,rooms,bedrooms,bathrooms,age_years,has_garage,has_balcony,has_pool,lat,lng,created_at
0,MLA757902287,Inmuebles,Alquiler,DEPARTAMENTO,Palermo,713000.0,50.0,50.0,NaN,2.0,1.0,14.0,False,True,False,-34.549232,-58.425510,2025-01-18 00:00:00
1,MLA371511639,Inmuebles,Alquiler,DEPARTAMENTO,Almagro,724000.0,50.0,50.0,2.0,1.0,1.0,NaN,False,False,False,-34.586467,-58.439921,2024-08-08 00:00:00
2,MLA154908801,Inmuebles,Alquiler,DEPARTAMENTO,Almagro,348000.0,42.0,NaN,2.0,NaN,1.0,13.0,True,False,False,-34.607493,-58.417589,2024-08-18 00:00:00
3,MLA403100436,Inmuebles,Alquiler,DEPARTAMENTO,Villa Urquiza,1217000.0,66.0,66.0,2.0,1.0,1.0,2.0,True,True,False,-34.567514,-58.487023,2024-11-03 00:00:00
4,MLA346000441,Inmuebles,Alquiler,CASA,Palermo,1121000.0,86.0,164.0,3.0,2.0,1.0,1.0,True,False,True,-34.589963,-58.444575,2025-03-22 00:00:00


## 8. Limpieza y normalización integral del Dataset B

In [12]:
# 1. Borrado y Renombrado inicial
cols_borrar_b = ['listing_id', 'category_l1', 'category_l2']
df_b.drop(columns=[c for c in cols_borrar_b if c in df_b.columns], inplace=True)

df_b.rename(columns={
    'property_type': 'tipo', 'neighborhood': 'barrio', 'price_ars': 'precio',
    'total_area_m2': 'superficie_total', 'rooms': 'ambientes',
    'age_years': 'antiguedad', 'has_garage': 'cochera',
    'has_balcony': 'balcon', 'has_pool': 'piscina',
    'lat': 'latitud', 'lng': 'longitud'
}, inplace=True)

# 2. Normalización de Tipo y Binarios
df_b['tipo'] = df_b['tipo'].astype(str).str.strip().str.capitalize()

for col in ['cochera', 'balcon', 'piscina']:
    if col in df_b.columns:
        df_b[col] = df_b[col].fillna(0).apply(lambda x: 1 if x != 0 and str(x).lower() not in ['no', 'nan', 'none'] else 0)

# 3. Superficie y Ambientes
df_b['superficie_total'] = pd.to_numeric(df_b['superficie_total'], errors='coerce')
if 'covered_area_m2' in df_b.columns:
    df_b['superficie_total'] = df_b['superficie_total'].fillna(pd.to_numeric(df_b['covered_area_m2'], errors='coerce'))

df_b['ambientes'] = pd.to_numeric(df_b['ambientes'], errors='coerce')
if 'bedrooms' in df_b.columns and 'bathrooms' in df_b.columns:
    df_b['ambientes'] = df_b['ambientes'].fillna(df_b['bedrooms'] + df_b['bathrooms'])

# Rellenos de seguridad para ambientes y superficie
df_b['superficie_total'] = df_b['superficie_total'].fillna(df_b.groupby('ambientes')['superficie_total'].transform('median')).fillna(df_b['superficie_total'].median())
df_b['ambientes'] = df_b['ambientes'].fillna(df_b['ambientes'].median()).round().astype('Int64')

# 4. Imputación de Antigüedad
# Limpiamos antigüedad primero para usarla como variable explicativa del precio
df_b['antiguedad'] = pd.to_numeric(df_b['antiguedad'], errors='coerce')
mediana_antig_barrio = df_b.groupby('barrio')['antiguedad'].transform('median')
df_b['antiguedad'] = df_b['antiguedad'].fillna(mediana_antig_barrio).fillna(df_b['antiguedad'].median()).round().astype('Int64')

# 5. Estimación de Precios
df_b['precio'] = pd.to_numeric(df_b['precio'], errors='coerce')

# Nivel 1: Máxima precisión (Barrio + Ambientes + Antigüedad + Adicionales)
columnas_ref_full = ['barrio', 'ambientes', 'antiguedad', 'cochera', 'balcon', 'piscina']
mediana_ultra = df_b.groupby(columnas_ref_full)['precio'].transform('median')

# Nivel 2: Referencia por Barrio + Ambientes + Adicionales
columnas_ref_std = ['barrio', 'ambientes', 'cochera', 'balcon', 'piscina']
mediana_pro = df_b.groupby(columnas_ref_std)['precio'].transform('median')

# Nivel 3: Valor de m2 por Barrio + Ambientes
df_b['precio_m2_temp'] = df_b['precio'] / df_b['superficie_total'].astype(float)
val_m2_barrio_amb = df_b.groupby(['barrio', 'ambientes'])['precio_m2_temp'].transform('median')

# Aplicamos los rellenos en cascada
df_b['precio'] = df_b['precio'].fillna(mediana_ultra)
df_b['precio'] = df_b['precio'].fillna(mediana_pro)
df_b['precio'] = df_b['precio'].fillna(val_m2_barrio_amb * df_b['superficie_total'].astype(float))
df_b['precio'] = df_b['precio'].fillna(df_b['precio'].median())

# 6. Limpieza y Formato Final
df_b.drop(columns=['precio_m2_temp', 'covered_area_m2', 'bedrooms', 'bathrooms'], errors='ignore', inplace=True)
df_b['precio'] = df_b['precio'].round().astype('Int64')
df_b['superficie_total'] = df_b['superficie_total'].round().astype('Int64')

orden_cols = ['barrio', 'tipo', 'precio', 'superficie_total', 'ambientes', 
              'antiguedad', 'cochera', 'balcon', 'piscina', 'latitud', 'longitud']
df_b = df_b[[c for c in orden_cols if c in df_b.columns]]

df_b.head()

,barrio,tipo,precio,superficie_total,ambientes,antiguedad,cochera,balcon,piscina,latitud,longitud
0,Palermo,Departamento,713000,50,3,14,0,1,0,-34.549232,-58.425510
1,Almagro,Departamento,724000,50,2,8,0,0,0,-34.586467,-58.439921
2,Almagro,Departamento,348000,42,2,13,1,0,0,-34.607493,-58.417589
3,Villa Urquiza,Departamento,1217000,66,2,2,1,1,0,-34.567514,-58.487023
4,Palermo,Casa,1121000,164,3,1,1,0,1,-34.589963,-58.444575


## 9. Limpieza y normalización del Dataset C

In [13]:
df_c.head()

,codigo,operacion,tipo_inmueble,barrio,precio_publicado,superficie,descripcion,coordenadas,publicado
0,AP-64391,Alquiler,ph,Villa Urquiza,$678.000/mes,55,2 ambientes. 1 dorm. 1 baños. pileta,"-34.57599599633878,-58.47863988110282",10-03-2024
1,AP-82260,Alquiler,ph,Colegiales,$700.000/mes,50,2 ambientes. 1 dorm. 1 baños. balcón. 11 años,"-34.582690398643514,-58.4524965859866",30-11-2024
2,AP-66765,Alquiler,departamento,Caballito,$719.000/mes,41,3 ambientes. 2 baños. 11 años,NaN,28-07-2024
3,AP-42570,Alquiler,departamento,Villa Crespo,$1.391.000/mes,58,3 ambientes. 2 dorm. con cochera. 4 años,NaN,31-12-2024
4,AP-64652,Alquiler,departamento,Recoleta,Consultar,53,3 ambientes. 2 dorm. 2 baños. con cochera. 18 ...,NaN,08-01-2025


## 10. Limpieza inicial del Dataset C

In [14]:
# 1. Borrado de columnas innecesarias
cols_a_eliminar = ['codigo', 'operacion']
df_c.drop(columns=[c for c in cols_a_eliminar if c in df_c.columns], inplace=True)

# 2. Renombrado de columnas al formato exacto de los otros datasets
df_c.rename(columns={
    'tipo_inmueble': 'tipo',
    'precio_publicado': 'precio',
    'superficie': 'superficie_total'
}, inplace=True)

# 3. Normalización de 'tipo' (Capitalizado: Departamento, Casa, Ph)
if 'tipo' in df_c.columns:
    df_c['tipo'] = df_c['tipo'].astype(str).str.strip().str.capitalize()

# 4. Reorganizar columnas
columnas_existentes = df_c.columns.tolist()
if 'barrio' in columnas_existentes and 'tipo' in columnas_existentes:
    orden_nuevo = ['barrio', 'tipo'] + [c for c in columnas_existentes if c not in ['barrio', 'tipo']]
    df_c = df_c[orden_nuevo]

df_c.head()

,barrio,tipo,precio,superficie_total,descripcion,coordenadas,publicado
0,Villa Urquiza,Ph,$678.000/mes,55,2 ambientes. 1 dorm. 1 baños. pileta,"-34.57599599633878,-58.47863988110282",10-03-2024
1,Colegiales,Ph,$700.000/mes,50,2 ambientes. 1 dorm. 1 baños. balcón. 11 años,"-34.582690398643514,-58.4524965859866",30-11-2024
2,Caballito,Departamento,$719.000/mes,41,3 ambientes. 2 baños. 11 años,NaN,28-07-2024
3,Villa Crespo,Departamento,$1.391.000/mes,58,3 ambientes. 2 dorm. con cochera. 4 años,NaN,31-12-2024
4,Recoleta,Departamento,Consultar,53,3 ambientes. 2 dorm. 2 baños. con cochera. 18 ...,NaN,08-01-2025


## 11. Desgloce de la columna "Descripción" (Dataset C)

In [ ]:
# 1. Aseguramos que la columna sea string para el procesamiento
df_c['descripcion'] = df_c['descripcion'].astype(str).str.lower()

# 2. Extracción del número de ambientes
df_c['ambientes'] = df_c['descripcion'].str.extract(r'(\d+)\s*amb').astype(float)

# 3. Auxiliares para calculos de baños y dormitorios
df_c['temp_baños'] = df_c['descripcion'].str.extract(r'(\d+)\s*baño').astype(float)
df_c['temp_dorm'] = df_c['descripcion'].str.extract(r'(\d+)\s*dorm').astype(float)

# Si 'ambientes' es nulo, sumamos dorm + baños
mask_amb_nulo = df_c['ambientes'].isna()
df_c.loc[mask_amb_nulo, 'ambientes'] = df_c['temp_dorm'] + df_c['temp_baños']

# 4. Extraer antiguedad (años)
df_c['antiguedad'] = df_c['descripcion'].str.extract(r'(\d+)\s*año').astype(float)
df_c.loc[df_c['descripcion'].str.contains('a estrenar'), 'antiguedad'] = 0

# 5. Extraer binarios (Cochera, Balcón, Piscina)
df_c['cochera'] = df_c['descripcion'].str.contains('cochera|garage').astype(int)
df_c['balcon'] = df_c['descripcion'].str.contains('balcón|balcon').astype(int)
df_c['piscina'] = df_c['descripcion'].str.contains('pileta|piscina').astype(int)

# 6. Borramos la descripción y las temporales de cálculo
df_c.drop(columns=['descripcion', 'temp_baños', 'temp_dorm'], inplace=True)

# 7. Formateo de tipos 
df_c['ambientes'] = df_c['ambientes'].fillna(df_c['ambientes'].median()).round().astype('Int64')
df_c['antiguedad'] = df_c['antiguedad'].fillna(df_c['antiguedad'].median()).round().astype('Int64')

df_c.head()

,barrio,tipo,precio,superficie_total,coordenadas,publicado,ambientes,antiguedad,cochera,balcon,piscina
0,Villa Urquiza,Ph,$678.000/mes,55,"-34.57599599633878,-58.47863988110282",10-03-2024,2,9,0,0,1
1,Colegiales,Ph,$700.000/mes,50,"-34.582690398643514,-58.4524965859866",30-11-2024,2,11,0,1,0
2,Caballito,Departamento,$719.000/mes,41,NaN,28-07-2024,3,11,0,0,0
3,Villa Crespo,Departamento,$1.391.000/mes,58,NaN,31-12-2024,3,4,1,0,0
4,Recoleta,Departamento,Consultar,53,NaN,08-01-2025,3,18,1,0,0


## 12. Limpieza de la columna "Superficie" (Dataset C)

In [16]:
def limpiar_rangos_superficie(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).lower().replace('m2', '').strip()
    
    # Si es un rango (ej: "58 a 68")
    if ' a ' in valor:
        try:
            partes = valor.split(' a ')
            num1 = float(partes[0].strip())
            num2 = float(partes[1].strip())
            return (num1 + num2) / 2  # promedio del rango
        except:
            return None
    
    # Si es un número solo
    try:
        return float(valor)
    except:
        return None

# 1. Aplicamos la limpieza de rangos
df_c['superficie_total'] = df_c['superficie_total'].apply(limpiar_rangos_superficie)

# 2. Rellenamos nulos usando la mediana según la cantidad de AMBIENTES
mediana_por_ambientes = df_c.groupby('ambientes')['superficie_total'].transform('median')
df_c['superficie_total'] = df_c['superficie_total'].fillna(mediana_por_ambientes)

# 3. Relleno residual (por si algún ambiente no tiene ninguna referencia)
df_c['superficie_total'] = df_c['superficie_total'].fillna(df_c['superficie_total'].median())

# 4. Formateamos a entero
df_c['superficie_total'] = df_c['superficie_total'].round().astype('Int64')

df_c.head()

,barrio,tipo,precio,superficie_total,coordenadas,publicado,ambientes,antiguedad,cochera,balcon,piscina
0,Villa Urquiza,Ph,$678.000/mes,55,"-34.57599599633878,-58.47863988110282",10-03-2024,2,9,0,0,1
1,Colegiales,Ph,$700.000/mes,50,"-34.582690398643514,-58.4524965859866",30-11-2024,2,11,0,1,0
2,Caballito,Departamento,$719.000/mes,41,NaN,28-07-2024,3,11,0,0,0
3,Villa Crespo,Departamento,$1.391.000/mes,58,NaN,31-12-2024,3,4,1,0,0
4,Recoleta,Departamento,Consultar,53,NaN,08-01-2025,3,18,1,0,0


## 13. Limpieza e Imputación inteligente de la columna "Precio ($)" (Dataset C)

In [17]:
# 1. Limpieza inicial de la columna precio (para identificar nulos reales)
def limpiar_precio_c(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).lower()
    if 'consultar' in valor:
        return None
    valor = valor.replace('$', '').replace('/mes', '').replace('.', '').strip()
    try:
        return float(valor)
    except:
        return None

df_c['precio'] = df_c['precio'].apply(limpiar_precio_c)

# 2. Asegurar binarios (necesarios para la segmentación de precios)
for col in ['cochera', 'balcon', 'piscina']:
    if col in df_c.columns:
        df_c[col] = df_c[col].fillna(0).apply(lambda x: 1 if x != 0 and str(x).lower() not in ['no', 'nan', 'none'] else 0)

# --- 3. Imputación de Antigüedad ---
# Usamos la mediana por barrio para que el precio se apoye en datos limpios
mediana_antig_barrio = df_c.groupby('barrio')['antiguedad'].transform('median')
df_c['antiguedad'] = df_c['antiguedad'].fillna(mediana_antig_barrio).fillna(df_c['antiguedad'].median()).round().astype('Int64')

# --- 4. Estimación de Precios ---
# Nivel 1: Máxima segmentación (Barrio + Ambientes + Antigüedad + Adicionales)
columnas_ultra = ['barrio', 'ambientes', 'antiguedad', 'cochera', 'balcon', 'piscina']
mediana_ultra = df_c.groupby(columnas_ultra)['precio'].transform('median')

# Nivel 2: Segmentación estándar (Barrio + Ambientes + Adicionales)
columnas_std = ['barrio', 'ambientes', 'cochera', 'balcon', 'piscina']
mediana_pro = df_c.groupby(columnas_std)['precio'].transform('median')

# Nivel 3: Mediana por m2 (Barrio + Ambientes)
df_c['precio_m2_temp'] = df_c['precio'] / df_c['superficie_total'].astype(float)
val_m2_barrio_amb = df_c.groupby(['barrio', 'ambientes'])['precio_m2_temp'].transform('median')

# Rellenamos en orden de precisión
df_c['precio'] = df_c['precio'].fillna(mediana_ultra)
df_c['precio'] = df_c['precio'].fillna(mediana_pro)
df_c['precio'] = df_c['precio'].fillna(val_m2_barrio_amb * df_c['superficie_total'].astype(float))
df_c['precio'] = df_c['precio'].fillna(df_c['precio'].median())

# 5. Formato Final y Limpieza
df_c['precio'] = df_c['precio'].round().astype('Int64')
df_c.drop(columns=['precio_m2_temp'], errors='ignore', inplace=True)

df_c.head()

,barrio,tipo,precio,superficie_total,coordenadas,publicado,ambientes,antiguedad,cochera,balcon,piscina
0,Villa Urquiza,Ph,678000,55,"-34.57599599633878,-58.47863988110282",10-03-2024,2,9,0,0,1
1,Colegiales,Ph,700000,50,"-34.582690398643514,-58.4524965859866",30-11-2024,2,11,0,1,0
2,Caballito,Departamento,719000,41,NaN,28-07-2024,3,11,0,0,0
3,Villa Crespo,Departamento,1391000,58,NaN,31-12-2024,3,4,1,0,0
4,Recoleta,Departamento,1120000,53,NaN,08-01-2025,3,18,1,0,0


## 14. Separación de Coordenadas (Dataset C)

In [ ]:
# 1. Extracción y Limpieza de Coordenadas Originales
if 'coordenadas' in df_c.columns:
    coords_split = df_c['coordenadas'].str.split(',', n=1, expand=True)
    df_c['latitud'] = pd.to_numeric(coords_split[0].str.strip(), errors='coerce')
    df_c['longitud'] = pd.to_numeric(coords_split[1].str.strip(), errors='coerce')
    df_c.drop(columns=['coordenadas'], inplace=True)

# 2. Imputación por Centroide de Barrio (Media de lat/lng por barrio)
# Calculamos la media y rellenamos en un solo paso
df_c['latitud'] = df_c['latitud'].fillna(df_c.groupby('barrio')['latitud'].transform('mean'))
df_c['longitud'] = df_c['longitud'].fillna(df_c.groupby('barrio')['longitud'].transform('mean'))

# 3. Relleno Residual y Formateo
# Si un barrio no tiene ninguna coordenada, usamos la media global del dataset
df_c['latitud'] = df_c['latitud'].fillna(df_c['latitud'].mean()).round(6)
df_c['longitud'] = df_c['longitud'].fillna(df_c['longitud'].mean()).round(6)

# 4. Reporte Final de Calidad de Datos
nulos_finales = df_c['latitud'].isna().sum()
print(f" Geolocalización finalizada: Coordenadas extraídas, imputadas por barrio y redondeadas.")
print(f"   Nulos restantes: {nulos_finales}")
print(df_c[['barrio', 'latitud', 'longitud']].head(3))

✅ Geolocalización finalizada: Coordenadas extraídas, imputadas por barrio y redondeadas.
   Nulos restantes: 0
          barrio    latitud   longitud
0  Villa Urquiza -34.575996 -58.478640
1     Colegiales -34.582690 -58.452497
2      Caballito -34.617906 -58.438543


## 15. Unificación Final: Datasets A, B y C

In [19]:
# 1. Concatenación inicial
df_propiedades = pd.concat([df_a, df_b, df_c], ignore_index=True)

# 2. Eliminación de Duplicados (Criterio Estricto)
# Solo borra si coinciden en casi todo
cols_duplicados = ['barrio', 'tipo', 'precio', 'superficie_total', 
                   'ambientes', 'antiguedad', 'cochera']

antes_dup = len(df_propiedades)
df_propiedades.drop_duplicates(subset=cols_duplicados, keep='first', inplace=True)
despues_dup = len(df_propiedades)

# 3. Eliminación de Outliers de Ambientes
mask_raros = (df_propiedades['tipo'] == 'Departamento') & (df_propiedades['ambientes'] > 6) & (df_propiedades['superficie_total'] < 130)
cantidad_raros = mask_raros.sum()
df_propiedades = df_propiedades[~mask_raros]

# 4. Ajuste final de nombres y tipos
df_propiedades['tipo'] = df_propiedades['tipo'].replace(['Ph', 'ph'], 'Ph')
df_propiedades['precio'] = df_propiedades['precio'].astype('Int64')

# 5. Orden de columnas final
columnas_orden = ['barrio', 'tipo', 'precio', 'superficie_total', 'ambientes', 
                  'antiguedad', 'cochera', 'balcon', 'piscina', 'latitud', 'longitud']
df_propiedades = df_propiedades[columnas_orden]

df_propiedades

,barrio,tipo,precio,superficie_total,ambientes,antiguedad,cochera,balcon,piscina,latitud,longitud
0,Belgrano,Departamento,1267000,75,4,4,1,1,0,-34.549646421652646,-58.462834001141694
1,Almagro,Departamento,584000,39,2,6,0,0,0,-34.60873815826056,-58.43909947532191
2,Flores,Departamento,252000,35,2,23,1,0,0,"-34,62205081312381",-58.45626733855754
3,Belgrano,Ph,546000,38,2,0,0,0,1,-34.574756771995716,-58.463757544997684
4,Villa Urquiza,Departamento,337000,33,1,2,0,1,0,-34.55428879305003,-58.4866156408398
...,...,...,...,...,...,...,...,...,...,...,...
10674,San Telmo,Departamento,998000,66,2,9,0,1,0,-34.621374,-58.369877
10675,Villa Crespo,Ph,943000,56,2,9,1,0,0,-34.600334,-58.435851
10676,Recoleta,Departamento,1579000,57,3,0,1,1,0,-34.587563,-58.395037
10677,Belgrano,Departamento,970000,51,2,32,1,0,0,-34.58208,-58.449371


## 16. Estadisticos descriptivos del Dataset Unificado

In [20]:
df_propiedades.describe()

,precio,superficie_total,ambientes,antiguedad,cochera,balcon,piscina
count,7819.0,7819.0,7819.0,7819.0,7819.000000,7819.000000,7819.000000
mean,1019557.408876,65.238266,2.643177,12.045914,0.346847,0.456069,0.114465
std,671340.849206,42.981942,1.276771,11.180171,0.475997,0.498098,0.318395
min,150000.0,20.0,1.0,0.0,0.000000,0.000000,0.000000
25%,573000.0,40.0,2.0,6.0,0.000000,0.000000,0.000000
50%,837000.0,55.0,2.0,9.0,0.000000,0.000000,0.000000
75%,1264500.0,77.0,3.0,13.0,1.000000,1.000000,0.000000
max,7712000.0,536.0,12.0,79.0,1.000000,1.000000,1.000000


## 17. Guardamos el Dataset Unificado

In [21]:
# 1. Limpieza de strings en coordenadas (Comas por Puntos)
for col in ['latitud', 'longitud']:
    df_propiedades[col] = (
        df_propiedades[col]
        .astype(str)
        .str.replace(',', '.')
        .pipe(pd.to_numeric, errors='coerce') # Convierte a float y pone NaN si algo falla
    )

# 2. Relleno de seguridad
df_propiedades['latitud'] = df_propiedades['latitud'].fillna(df_propiedades['latitud'].mean())
df_propiedades['longitud'] = df_propiedades['longitud'].fillna(df_propiedades['longitud'].mean())

# 3. Definir la ruta y asegurar carpeta
output_path = "../output/dataset_unificado.parquet"
os.makedirs("../output", exist_ok=True)

# 4. Guardar
try:
    df_propiedades.to_parquet(output_path, engine='pyarrow', index=False)
    print(f"Archivo guardado en: {output_path}")
    print(f"Dataset finalizado con {len(df_propiedades)} filas.")
except Exception as e:
    print(f"❌ Error al guardar: {e}")

Archivo guardado en: ../output/dataset_unificado.parquet
Dataset finalizado con 7819 filas.
